In [31]:
import c45
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder


Rodando C45 antes do balanceamento

In [32]:
modelo = c45.C45Classifier(max_depth=6, min_samples_split=10)

In [33]:
print("oi")
df = pd.read_parquet('dados_filtradoseprocessados.parquet')

oi


In [34]:

df.info()
df = df.drop(['M00203','P036'], axis=1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53630 entries, 0 to 53629
Data columns (total 28 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   A001     53630 non-null  object 
 1   C001     53630 non-null  float64
 2   C006     53630 non-null  object 
 3   C008     53630 non-null  object 
 4   C009     53630 non-null  object 
 5   C011     53630 non-null  object 
 6   D001     53630 non-null  object 
 7   M00203   53630 non-null  object 
 8   M005010  28477 non-null  object 
 9   M01601   53630 non-null  object 
 10  N010     53630 non-null  object 
 11  N011     53630 non-null  object 
 12  N016     53630 non-null  object 
 13  N017     53630 non-null  float64
 14  P02002   53630 non-null  float64
 15  P02601   53630 non-null  object 
 16  P02602   53630 non-null  float64
 17  P02801   53630 non-null  float64
 18  P034     53630 non-null  object 
 19  P036     53630 non-null  object 
 20  P038     53630 non-null  object 
 21  P039     536

In [35]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
numeric_cols = numeric_cols.drop('C001')
def classificar_dieta(valor):
    """
    Retorna 'Incluso na dieta' se valor != 0
    Retorna 'Não incluso na dieta' se valor == 0
    """
    if valor == 0:
        return "Nao incluso na dieta"
    else:
        return "Incluso na dieta"
df['VDM001'] = df['VDM001'].apply(classificar_dieta)
for coluna in numeric_cols:
    df[f'Status_{coluna}'] = df[coluna].apply(classificar_dieta)



df = df.drop(numeric_cols, axis=1)
df2 = df.copy()

In [36]:


encoder = OneHotEncoder()
ordinal_cols = ['VDF004', 'VDD004A', 'M01601', 'VDM001', 'P02601', 'N010', 'N011', 'N016']
categorical_cols = df.select_dtypes(include=['object', 'category']).columns   

# Remova o inplace=True
categorical_cols = categorical_cols.drop(['VDF004', 'VDD004A'])

label_encoder = LabelEncoder()

for col in ordinal_cols:
    df[col] = label_encoder.fit_transform(df[col])

df_encoded = pd.get_dummies(df, drop_first=True, columns=categorical_cols)

df_encoded.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53630 entries, 0 to 53629
Data columns (total 48 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   C001                                         53630 non-null  float64
 1   VDD004A                                      53630 non-null  int64  
 2   VDF004                                       53630 non-null  int64  
 3   A001_Casa                                    53630 non-null  bool   
 4   A001_Habitacao em casa de comodos (cortico)  53630 non-null  bool   
 5   C006_Mulher                                  53630 non-null  bool   
 6   C008_50-59 anos                              53630 non-null  bool   
 7   C008_60-69 anos                              53630 non-null  bool   
 8   C008_70-79 anos                              53630 non-null  bool   
 9   C008_80-89 anos                              53630 non-null  bool   
 10

,A001,C001,C006,C008,C009,C011,D001,M005010,M01601,N010,...,P044,P046,Q00201,VDD004A,VDF004,VDM001,Status_N017,Status_P02002,Status_P02602,Status_P02801
0,Casa,6.0,Mulher,50-59 anos,Branca,Casado,Sim,Nao,3,3,...,Nao,Sim,Sim,1,3,0,Incluso na dieta,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta
1,Casa,8.0,Mulher,39-49 anos,Preta,Solteiro,Sim,Nao,3,2,...,Nao,Nao,Nao,2,2,0,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta,Incluso na dieta
2,Casa,1.0,Mulher,50-59 anos,Preta,Viuvo,Nao,Nao,5,2,...,Sim,Sim,Nao,4,3,0,Incluso na dieta,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta
3,Casa,1.0,Homem,80-89 anos,Parda,Solteiro,Sim,None,3,3,...,Nao,Nao,Sim,1,2,0,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta,Nao incluso na dieta
4,Casa,3.0,Mulher,39-49 anos,Parda,Casado,Sim,Nao,2,2,...,Sim,Sim,Nao,2,1,0,Incluso na dieta,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53625,Casa,2.0,Mulher,50-59 anos,Branca,Solteiro,Sim,Nao,0,3,...,Sim,Nao,Nao,3,1,0,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta,Incluso na dieta
53626,Casa,5.0,Homem,39-49 anos,Preta,Solteiro,Sim,Nao,3,1,...,Nao,Nao,Nao,4,3,0,Incluso na dieta,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta
53627,Casa,3.0,Mulher,50-59 anos,Parda,Casado,Sim,Nao,3,2,...,Nao,Nao,Nao,2,1,0,Incluso na dieta,Incluso na dieta,Nao incluso na dieta,Incluso na dieta
53628,Casa,2.0,Homem,39-49 anos,Parda,Divorciado/separado,Sim,Nao,3,1,...,Nao,Sim,Nao,2,1,0,Incluso na dieta,Nao incluso na dieta,Nao incluso na dieta,Incluso na dieta


In [37]:
x = df_encoded.drop('Q00201_Sim', axis=1)
y = df_encoded['Q00201_Sim']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


modelo.fit(x_train, y_train)
y_pred = modelo.predict(x_test)

In [38]:
y_pred
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[5561  966]
 [2620 1579]]
              precision    recall  f1-score   support

       False       0.68      0.85      0.76      6527
        True       0.62      0.38      0.47      4199

    accuracy                           0.67     10726
   macro avg       0.65      0.61      0.61     10726
weighted avg       0.66      0.67      0.64     10726



In [39]:

def exportar_csv(df, caminho_arquivo, index=False, encoding='utf-8', sep=','):
    """
    Exporta um DataFrame pandas para arquivo CSV.
    
    Parâmetros:
    -----------
    df : pd.DataFrame
        DataFrame a ser exportado
    caminho_arquivo : str
        Caminho completo do arquivo (ex: 'dados/resultado.csv')
    index : bool, default=False
        Se True, inclui o índice no CSV
    encoding : str, default='utf-8'
        Codificação do arquivo (use 'utf-8-sig' para Excel)
    sep : str, default=','
        Separador de colunas (use ';' para padrão brasileiro)
    
    Retorna:
    --------
    str : Mensagem de confirmação
    """
    try:
        df.to_csv(caminho_arquivo, index=index, encoding=encoding, sep=sep)
        return f"✓ Arquivo salvo com sucesso em: {caminho_arquivo}"
    except Exception as e:
        return f"✗ Erro ao salvar arquivo: {str(e)}"

# Exemplo de uso:
# df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
# exportar_csv(df, 'meus_dados.csv')
# exportar_csv(df, 'dados.csv', sep=';', encoding='utf-8-sig')  # Para Excel


exportar_csv(df_encoded, 'dados_c45_sembalanceamento.csv', index=False, encoding='utf-8-sig', sep=';')
exportar_csv(df2, 'dados_c45_sembalanceamento_originais.csv', index=False, encoding='utf-8-sig', sep=';')

'✓ Arquivo salvo com sucesso em: dados_c45_sembalanceamento_originais.csv'

In [40]:
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

# Aplicar undersampling nos dados de treino
print("Distribuição original:")
print(Counter(y_train))

rus = RandomUnderSampler(random_state=42)
x_train_under, y_train_under = rus.fit_resample(x_train, y_train)

print("\nDistribuição após undersampling:")
print(Counter(y_train_under))

# Treinar modelo com dados balanceados
modelo2 = c45.C45Classifier(max_depth=6, min_samples_split=10)
modelo2.fit(x_train_under, y_train_under)
y_pred2 = modelo2.predict(x_test)

print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, y_pred2))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred2))

Distribuição original:
Counter({False: 26530, True: 16374})

Distribuição após undersampling:
Counter({False: 16374, True: 16374})

Matriz de Confusão:
[[4619 1908]
 [1845 2354]]

Relatório de Classificação:
              precision    recall  f1-score   support

       False       0.71      0.71      0.71      6527
        True       0.55      0.56      0.56      4199

    accuracy                           0.65     10726
   macro avg       0.63      0.63      0.63     10726
weighted avg       0.65      0.65      0.65     10726



In [41]:
from imblearn.over_sampling import SMOTE


x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


# Aplicar SMOTE nos dados de treino
print("Distribuição original:")
print(Counter(y_train))

smote = SMOTE(random_state=42)
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

print("\nDistribuição após SMOTE:")
print(Counter(y_train_smote))

# Treinar modelo com dados balanceados
modelo3 = c45.C45Classifier(max_depth=6, min_samples_split=10)
modelo3.fit(x_train_smote, y_train_smote)
y_pred3 = modelo3.predict(x_test)

print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, y_pred3))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred3))

Distribuição original:
Counter({False: 26530, True: 16374})

Distribuição após SMOTE:
Counter({True: 26530, False: 26530})

Matriz de Confusão:
[[4618 1909]
 [1846 2353]]

Relatório de Classificação:
              precision    recall  f1-score   support

       False       0.71      0.71      0.71      6527
        True       0.55      0.56      0.56      4199

    accuracy                           0.65     10726
   macro avg       0.63      0.63      0.63     10726
weighted avg       0.65      0.65      0.65     10726



In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53630 entries, 0 to 53629
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   A001           53630 non-null  object 
 1   C001           53630 non-null  float64
 2   C006           53630 non-null  object 
 3   C008           53630 non-null  object 
 4   C009           53630 non-null  object 
 5   C011           53630 non-null  object 
 6   D001           53630 non-null  object 
 7   M005010        28477 non-null  object 
 8   M01601         53630 non-null  int64  
 9   N010           53630 non-null  int64  
 10  N011           53630 non-null  int64  
 11  N016           53630 non-null  int64  
 12  P02601         53630 non-null  int64  
 13  P034           53630 non-null  object 
 14  P038           53630 non-null  object 
 15  P039           53630 non-null  object 
 16  P044           53630 non-null  object 
 17  P046           53630 non-null  object 
 18  Q00201

In [43]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53630 entries, 0 to 53629
Data columns (total 48 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   C001                                         53630 non-null  float64
 1   VDD004A                                      53630 non-null  int64  
 2   VDF004                                       53630 non-null  int64  
 3   A001_Casa                                    53630 non-null  bool   
 4   A001_Habitacao em casa de comodos (cortico)  53630 non-null  bool   
 5   C006_Mulher                                  53630 non-null  bool   
 6   C008_50-59 anos                              53630 non-null  bool   
 7   C008_60-69 anos                              53630 non-null  bool   
 8   C008_70-79 anos                              53630 non-null  bool   
 9   C008_80-89 anos                              53630 non-null  bool   
 10